<a href="https://colab.research.google.com/github/harneet862/ur2phd-assignment/blob/main/Part3_llm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q transformers accelerate
!git clone https://github.com/ilopezgazpio/SemEval-2027-Task-2-DiCo-NLI.git

Cloning into 'SemEval-2027-Task-2-DiCo-NLI'...
remote: Enumerating objects: 518, done.
remote: Counting objects: 100% (47/47), done.
remote: Compressing objects: 100% (5/5), done.
remote: Total 518 (delta 45), reused 42 (delta 42), pack-reused 471 (from 1)
Receiving objects: 100% (518/518), 29.85 MiB | 23.97 MiB/s, done.
Resolving deltas: 100% (268/268), done.


In [ ]:
import pandas as pd, os, torch

TASK_DIR = "/content/SemEval-2027-Task-2-DiCo-NLI"
RESULTS_DIR = f"{TASK_DIR}/results"
os.makedirs(RESULTS_DIR, exist_ok=True)

train = pd.read_csv(f"{TASK_DIR}/final_data/train/dico_nli_train_track1_participant_labeled.csv")
dev = pd.read_csv(f"{TASK_DIR}/final_data/dev/dico_nli_dev_track1_reference.csv")
print(len(train), len(dev))

3042 660


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

LLM_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
# getting teh tokeniser for the given llm to convert the words to numbers
llm_tok = AutoTokenizer.from_pretrained(LLM_NAME)
# getting the left to right model
llm = AutoModelForCausalLM.from_pretrained(LLM_NAME, torch_dtype=torch.float16, device_map="cuda")
# to turn off dropout which is so that no sensors are turned off later on (safety habit)
llm.eval()

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 1536)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=1536, out_features=1536, bias=True)
          (k_proj): Linear(in_features=1536, out_features=256, bias=True)
          (v_proj): Linear(in_features=1536, out_features=256, bias=True)
          (o_proj): Linear(in_features=1536, out_features=1536, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (up_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (down_proj): Linear(in_features=8960, out_features=1536, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((1536,), eps=1e-06)
    (rotar

## Zero shot - no examples just explanation of how to classify?

In [ ]:
ZERO_SHOT_SYSTEM_PROMPT = """You classify the relation between two short English phrases, text1 and text2.
Choose exactly one label:
EQUIVALENCE: text1 and text2 mean the same thing.
FORWARD_ENTAILMENT: if text1 is true, text2 must also be true, but not the other way around.
BACKWARD_ENTAILMENT: if text2 is true, text1 must also be true, but not the other way around.
NEGATIVE_OTHER: none of the above.
Reply with the label only."""

In [ ]:
def build_messages(text1, text2, SYSTEM_PROMPT):
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"text1: {text1}\ntext2: {text2}\nLabel:"},
    ]

def generate(messages):
    # turn the chat messages into the exact text format Qwen was trained on
    prompt = llm_tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    # turn this into number tensors and move to cuda
    inputs = llm_tok(prompt, return_tensors="pt").to(llm.device)
    with torch.no_grad():
        out = llm.generate(**inputs, max_new_tokens=10, do_sample=False)
    # keep only the newly generated tokens, not the prompt
    new_tokens = out[0][inputs["input_ids"].shape[1]:]
    return llm_tok.decode(new_tokens, skip_special_tokens=True)


In [ ]:
messages =[{"role": "system", "content": ZERO_SHOT_SYSTEM_PROMPT},{"role": "user", "content": "text1: hello\ntext2: hello everyone\nLabel:"}]
# this adds the tokens which the llm requires like the start and end
prompt = llm_tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
print(prompt)
# converts it to numbers
inputs = llm_tok(prompt, return_tensors="pt").to(llm.device)
print(inputs)


<|im_start|>system
You classify the relation between two short English phrases, text1 and text2.
Choose exactly one label:
EQUIVALENCE: text1 and text2 mean the same thing.
FORWARD_ENTAILMENT: if text1 is true, text2 must also be true, but not the other way around.
BACKWARD_ENTAILMENT: if text2 is true, text1 must also be true, but not the other way around.
NEGATIVE_OTHER: none of the above.
Reply with the label only.<|im_end|>
<|im_start|>user
text1: hello
text2: hello everyone
Label:<|im_end|>
<|im_start|>assistant

{'input_ids': tensor([[151644,   8948,    198,   2610,  48129,    279,  12687,   1948,   1378,
           2805,   6364,  31747,     11,   1467,     16,    323,   1467,     17,
            624,  24051,   6896,    825,   2383,    510,     36,  11979,  12820,
          25317,   2104,     25,   1467,     16,    323,   1467,     17,   3076,
            279,   1852,   3166,    624,  30902,  28166,  16241,   5965,   5328,
             25,    421,   1467,     16,    374,    830, 

In [ ]:
for i in range(5):
    row = dev.iloc[i]
    raw = generate(build_messages(row["text1"], row["text2"], ZERO_SHOT_SYSTEM_PROMPT))
    print(raw)


EQUIVALENCE
'EQUIVALENCE' | gold: FORWARD_ENTAILMENT | 's war shrine visit -> 's shrine visit
FORWARD_ENTAILMENT
'FORWARD_ENTAILMENT' | gold: BACKWARD_ENTAILMENT | 's shrine visit -> 's war shrine visit
EQUIVALENCE
'EQUIVALENCE' | gold: FORWARD_ENTAILMENT | anti-protest laws -> anti-protest law
EQUIVALENCE
'EQUIVALENCE' | gold: BACKWARD_ENTAILMENT | anti-protest law -> anti-protest laws
EQUIVALENCE
'EQUIVALENCE' | gold: FORWARD_ENTAILMENT | Iran nuclear experts and also world powers -> World powers and Iran


In [ ]:
import re

LABELS = ["EQUIVALENCE", "FORWARD_ENTAILMENT", "BACKWARD_ENTAILMENT", "NEGATIVE_OTHER"]
FALLBACK = "NEGATIVE_OTHER"

# matches any of the 4 labels
LABEL_PATTERN = re.compile(r"EQUIVALENCE|FORWARD_ENTAILMENT|BACKWARD_ENTAILMENT|NEGATIVE_OTHER")

def parse_label(raw):
    # normalise: uppercase, and turn spaces/hyphens into underscores
    # so "forward entailment" and "Forward-Entailment" both become FORWARD_ENTAILMENT
    text = re.sub(r"[\s\-]+", "_", raw.strip().upper())
    match = LABEL_PATTERN.search(text)
    if match:
        return match.group(0), True    # (label, parsed successfully)
    return FALLBACK, False             # (fallback label, not parsed)

## now we get each row of the dev and then pass it to the model to make prediction and then pass it through the parser

In [ ]:
raw_outputs = []
pred_labels = []
parsed_ok = []

for i, row in dev.iterrows():
    messages = build_messages(row["text1"], row["text2"], ZERO_SHOT_SYSTEM_PROMPT)
    output = generate(messages)
    label, ok = parse_label(output)

    raw_outputs.append(output)
    pred_labels.append(label)
    parsed_ok.append(ok)

dev["raw_zeroshot"] = raw_outputs
dev["pred_zeroshot"] = pred_labels
dev["parsed_ok_zeroshot"] = parsed_ok

print("Rows:", len(dev))
print("Unparsed:", (~dev["parsed_ok_zeroshot"]).sum())
print(dev["pred_zeroshot"].value_counts())

Rows: 660
Unparsed: 0
pred_zeroshot
FORWARD_ENTAILMENT     348
EQUIVALENCE            270
NEGATIVE_OTHER          41
BACKWARD_ENTAILMENT      1
Name: count, dtype: int64


In [ ]:
print(set(pred_labels))

{'NEGATIVE_OTHER', 'BACKWARD_ENTAILMENT', 'FORWARD_ENTAILMENT', 'EQUIVALENCE'}


In [ ]:
# predictions file for the scorer: only instance_id and label
pd.DataFrame({
    "instance_id": dev["instance_id"],
    "label": dev["pred_zeroshot"],
}).to_csv(f"{RESULTS_DIR}/predictions_llm_zeroshot.csv", index=False)

dev[["instance_id", "text1", "text2", "label", "raw_zeroshot", "pred_zeroshot", "parsed_ok_zeroshot"]].to_csv(
    f"{RESULTS_DIR}/llm_zeroshot_raw.csv", index=False)



In [ ]:
%cd /content/SemEval-2027-Task-2-DiCo-NLI
!python3 -m evaluation_functions \
  --gold final_data/dev/dico_nli_dev_track1_reference.csv \
  --predictions results/predictions_llm_zeroshot.csv \
  --output-dir results/llm_zeroshot
!cat results/llm_zeroshot/scores.txt

/content/SemEval-2027-Task-2-DiCo-NLI
{
  "classification": {
    "accuracy": 0.2833333333333333,
    "confusion_matrix": {
      "BACKWARD_ENTAILMENT": {
        "BACKWARD_ENTAILMENT": 0,
        "EQUIVALENCE": 47,
        "FORWARD_ENTAILMENT": 133,
        "NEGATIVE_OTHER": 10
      },
      "EQUIVALENCE": {
        "BACKWARD_ENTAILMENT": 0,
        "EQUIVALENCE": 92,
        "FORWARD_ENTAILMENT": 78,
        "NEGATIVE_OTHER": 4
      },
      "FORWARD_ENTAILMENT": {
        "BACKWARD_ENTAILMENT": 1,
        "EQUIVALENCE": 109,
        "FORWARD_ENTAILMENT": 74,
        "NEGATIVE_OTHER": 6
      },
      "NEGATIVE_OTHER": {
        "BACKWARD_ENTAILMENT": 0,
        "EQUIVALENCE": 22,
        "FORWARD_ENTAILMENT": 63,
        "NEGATIVE_OTHER": 21
      }
    },
    "macro_f1": 0.2438054092329185,
    "per_label": {
      "BACKWARD_ENTAILMENT": {
        "f1": 0.0,
        "false_negative": 190,
        "false_positive": 1,
        "label": "BACKWARD_ENTAILMENT",
        "precision": 0.

In [ ]:
!cd /content/SemEval-2027-Task-2-DiCo-NLI && zip -r /content/zero_shot_results_part3.zip results
from google.colab import files
files.download("/content/zero_shot_results_part3.zip")

updating: results/ (stored 0%)
updating: results/llm_zeroshot/ (stored 0%)
updating: results/llm_zeroshot/scores.json (deflated 94%)
updating: results/llm_zeroshot/scores.txt (deflated 13%)
updating: results/llm_zeroshot_raw.csv (deflated 88%)
updating: results/predictions_llm_zeroshot.csv (deflated 93%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## now few shot prompting - adding examples to the zero_shot prompt

In [ ]:
fewshot_examples = (
    train.groupby("label", group_keys=False)
         .apply(lambda g: g.sample(2, random_state=0))
)
print(fewshot_examples[["instance_id", "text1", "text2", "label"]].to_string())

                                instance_id                                       text1                                       text2                label
301   dico_phrasis_0000216__en-en__original                                       A boy                                 A young boy  BACKWARD_ENTAILMENT
813   dico_phrasis_0000584__en-en__original      with a bracelet a bikini top and jeans  jeans , a dark bikini top , and a bracelet  BACKWARD_ENTAILMENT
898    dico_phrasis_0000642__en-en__flipped                          Ukraine protesters                        Ukrainian protesters          EQUIVALENCE
1143  dico_phrasis_0000841__en-en__original                              to another man                         toward a second man          EQUIVALENCE
300    dico_phrasis_0000216__en-en__flipped                                 A young boy                                       A boy   FORWARD_ENTAILMENT
812    dico_phrasis_0000584__en-en__flipped  jeans , a dark bikini top , and a bra

/tmp/ipykernel_2246/3961626713.py:3: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(2, random_state=0))


In [ ]:
examples_text = "\n\n".join(
    f"text1: {r['text1']}\ntext2: {r['text2']}\nLabel: {r['label']}"
    for _, r in fewshot_examples.iterrows()
)

FEW_SHOT_SYSTEM_PROMPT = ZERO_SHOT_SYSTEM_PROMPT + "\n\nExamples:\n\n" + examples_text
print(FEW_SHOT_SYSTEM_PROMPT)

You classify the relation between two short English phrases, text1 and text2.
Choose exactly one label:
EQUIVALENCE: text1 and text2 mean the same thing.
FORWARD_ENTAILMENT: if text1 is true, text2 must also be true, but not the other way around.
BACKWARD_ENTAILMENT: if text2 is true, text1 must also be true, but not the other way around.
NEGATIVE_OTHER: none of the above.
Reply with the label only.

Examples:

text1: A boy
text2: A young boy
Label: BACKWARD_ENTAILMENT

text1: with a bracelet a bikini top and jeans
text2: jeans , a dark bikini top , and a bracelet
Label: BACKWARD_ENTAILMENT

text1: Ukraine protesters
text2: Ukrainian protesters
Label: EQUIVALENCE

text1: to another man
text2: toward a second man
Label: EQUIVALENCE

text1: A young boy
text2: A boy
Label: FORWARD_ENTAILMENT

text1: jeans , a dark bikini top , and a bracelet
text2: with a bracelet a bikini top and jeans
Label: FORWARD_ENTAILMENT

text1: Russian Prime Minister
text2: Russia
Label: NEGATIVE_OTHER

text1: by

In [ ]:
raw_outputs = []
pred_labels = []
parsed_ok = []

for i, row in dev.iterrows():
    messages = build_messages(row["text1"], row["text2"], FEW_SHOT_SYSTEM_PROMPT)
    output = generate(messages)
    label, ok = parse_label(output)

    raw_outputs.append(output)
    pred_labels.append(label)
    parsed_ok.append(ok)

dev["raw_fewshot"] = raw_outputs
dev["pred_fewshot"] = pred_labels
dev["parsed_ok_fewshot"] = parsed_ok

print("Rows:", len(dev))
print("Unparsed:", (~dev["parsed_ok_fewshot"]).sum())
print(dev["pred_fewshot"].value_counts())

Rows: 660
Unparsed: 0
pred_fewshot
EQUIVALENCE            306
FORWARD_ENTAILMENT     261
NEGATIVE_OTHER          82
BACKWARD_ENTAILMENT     11
Name: count, dtype: int64


In [ ]:
print(set(pred_labels))

{'NEGATIVE_OTHER', 'BACKWARD_ENTAILMENT', 'FORWARD_ENTAILMENT', 'EQUIVALENCE'}


In [ ]:
# predictions file for the scorer: only instance_id and label
pd.DataFrame({
    "instance_id": dev["instance_id"],
    "label": dev["pred_fewshot"],
}).to_csv(f"{RESULTS_DIR}/predictions_llm_fewshot.csv", index=False)

dev[["instance_id", "text1", "text2", "label", "raw_fewshot", "pred_fewshot", "parsed_ok_fewshot"]].to_csv(
    f"{RESULTS_DIR}/llm_fewshot_raw.csv", index=False)



In [ ]:
%cd /content/SemEval-2027-Task-2-DiCo-NLI
!python3 -m evaluation_functions \
  --gold final_data/dev/dico_nli_dev_track1_reference.csv \
  --predictions results/predictions_llm_fewshot.csv \
  --output-dir results/llm_fewshot
!cat results/llm_fewshot/scores.txt

/content/SemEval-2027-Task-2-DiCo-NLI
{
  "classification": {
    "accuracy": 0.31212121212121213,
    "confusion_matrix": {
      "BACKWARD_ENTAILMENT": {
        "BACKWARD_ENTAILMENT": 2,
        "EQUIVALENCE": 68,
        "FORWARD_ENTAILMENT": 97,
        "NEGATIVE_OTHER": 23
      },
      "EQUIVALENCE": {
        "BACKWARD_ENTAILMENT": 7,
        "EQUIVALENCE": 104,
        "FORWARD_ENTAILMENT": 55,
        "NEGATIVE_OTHER": 8
      },
      "FORWARD_ENTAILMENT": {
        "BACKWARD_ENTAILMENT": 2,
        "EQUIVALENCE": 109,
        "FORWARD_ENTAILMENT": 64,
        "NEGATIVE_OTHER": 15
      },
      "NEGATIVE_OTHER": {
        "BACKWARD_ENTAILMENT": 0,
        "EQUIVALENCE": 25,
        "FORWARD_ENTAILMENT": 45,
        "NEGATIVE_OTHER": 36
      }
    },
    "macro_f1": 0.28000657536960194,
    "per_label": {
      "BACKWARD_ENTAILMENT": {
        "f1": 0.01990049751243781,
        "false_negative": 188,
        "false_positive": 9,
        "label": "BACKWARD_ENTAILMENT",
    

In [ ]:
!cd /content/SemEval-2027-Task-2-DiCo-NLI && zip -r /content/few_shot_results_part3.zip results
from google.colab import files
files.download("/content/few_shot_results_part3.zip")

  adding: results/ (stored 0%)
  adding: results/llm_zeroshot/ (stored 0%)
  adding: results/llm_zeroshot/scores.json (deflated 94%)
  adding: results/llm_zeroshot/scores.txt (deflated 13%)
  adding: results/llm_zeroshot_raw.csv (deflated 88%)
  adding: results/llm_fewshot_raw.csv (deflated 88%)
  adding: results/llm_fewshot/ (stored 0%)
  adding: results/llm_fewshot/scores.json (deflated 94%)
  adding: results/llm_fewshot/scores.txt (deflated 10%)
  adding: results/predictions_llm_fewshot.csv (deflated 93%)
  adding: results/predictions_llm_zeroshot.csv (deflated 93%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>